# **Step 1: Setup Hugging Face API Token**

# **Step 1: Install Required Dependencies**

In [ ]:
!pip install unsloth # install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git # Also get the latest version Unsloth!

# **Step 2: Import necessary Libraries**

In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from unsloth import is_bfloat16_supported
from huggingface_hub import login
from transformers import TrainingArguments
from datasets import load_dataset
import wandb

# **Step 4: Check Hugging Face Token**

In [ ]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

# **GPU Is Available?**

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

# **Step 5: Pretrained Deepseek R1**

In [ ]:
model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
max_sequence_length = 1024
dtype = None
load_in_4bit = True

# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=model_name,
#     # max_sequence_length=max_sequence_length,
#     dtype = dtype,
#     load_in_4bit=load_in_4bit,
#     token = hf_token
# )

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    load_in_4bit = load_in_4bit,
    max_seq_length = 1024,
    fast_inference = False,
    disable_log_stats = True,
    local_files_only = False,
)


# **Step 6: Setup System Prompt**

In [ ]:

prompt_style = """
Below is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response that effectively addresses the request.

Before crafting your answer, take a moment to carefully analyze the question. Develop a clear, step-by-step thought process to ensure your response is both logical and accurate.

### Task:
You are a medical expert specializing in clinical reasoning, diagnostics, and treatment planning. Answer the medical question below using your advanced knowledge.

### Query:
{}

### Answer:
<think>{}
"""

# **Step 7: Model Inferencing**

In [ ]:
# import warnings
# warnings.filterwarnings("ignore")

# from torch.nn import attention
# # Define a test question

# prompt = f"""<|user|> A 61-year-old woman with a long history of involuntary urine loss during activities like coughing or
#               sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings,
#               what would cystometry most likely reveal about her residual volume and detrusor contractions?"""

# # FastLanguageModel.for_inference(model)

# # Tokenize the Input
# inputs = tokenizer(
#     prompt,
#     return_tensors="pt",
#     padding=True,
#     truncation=True
# ).to("cuda")

# # Generate a Response
# outputs = model.generate(
#     input_ids=inputs.input_ids,
#     attention_mask=inputs.attention_mask,
#     max_new_tokens=200,
#     pad_token_id=tokenizer.eos_token_id,
#     use_cache=True
# )


# # Decode the response tokens back to text
# response = tokenizer.batch_decode(outputs)

# # Print the response
# print(response)

In [ ]:
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = model_name,
#     load_in_4bit = True,
#     max_seq_length = 2048,
#     fast_inference = False,
# )

model = model.eval()   # ✅ important

prompt = """<|user|>
A 61-year-old woman with a long history of involuntary urine loss during activities like coughing or sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings, what would cystometry most likely reveal about her residual volume and detrusor contractions?


<|assistant|>
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    padding=True,
    truncation=True
).to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=200,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=True,
    temperature=0.7,
    use_cache=False   # 🔥 KEY FIX
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True   # ✅ important
)

response = response.replace("Ġ", " ")

print(response)

In [ ]:
print(response.split('<|assistant|>')[1].lstrip().replace('ĊĊ', '\n\n'))

In [ ]:
import re

# The `prompt` variable is available from the previous execution, containing the user's input for generation.
# Extract the actual user query part from the `prompt` variable, which is known to be well-formatted.
user_query_text = prompt.split('<|user|>')[1].split('<|assistant|>')[0].strip()
# Clean up any potential extra newlines or multiple spaces that might have been part of the prompt string's formatting
user_query_text = re.sub(r'\n\s*', ' ', user_query_text).strip()
user_query_text = re.sub(r' +', ' ', user_query_text).strip()


# Decode the full raw output from the model
# We set clean_up_tokenization_spaces=False here, as we will handle spaces manually more precisely
raw_decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=False, clean_up_tokenization_spaces=False)

# Remove the beginning of sentence token which often appears as <\uff5cbegin of sentence\uff5c>
raw_decoded_output = raw_decoded_output.replace('<｜begin of sentence｜>', '').strip()

# Split the raw decoded output to isolate the assistant's generated part
parts_of_decoded_output = raw_decoded_output.split('<|assistant|>', 1) # Split only once

cleaned_assistant_response = ""
if len(parts_of_decoded_output) > 1:
    assistant_raw_content = parts_of_decoded_output[1]

    # Clean up the assistant's content:
    # 1. Replace 'Ġ' tokens with actual spaces
    cleaned_assistant_response = assistant_raw_content.replace('Ġ', ' ')

    # 2. Apply regex to insert spaces where words or numbers might have been merged in the assistant's output
    cleaned_assistant_response = re.sub(r'([a-z])([A-Z])', r'\1 \2', cleaned_assistant_response)
    cleaned_assistant_response = re.sub(r'([a-zA-Z])([0-9])', r'\1 \2', cleaned_assistant_response)
    cleaned_assistant_response = re.sub(r'([0-9])([a-zA-Z])', r'\1 \2', cleaned_assistant_response)

    # 3. Replace 'ĊĊ' with actual newlines
    cleaned_assistant_response = cleaned_assistant_response.replace('ĊĊ', '\n\n').strip()

    # 4. Remove any extra spaces introduced by previous replacements
    cleaned_assistant_response = re.sub(r' +', ' ', cleaned_assistant_response).strip()


# Reconstruct the full output using the clean user query and the cleaned assistant response
final_full_output = f"<|user|> {user_query_text}\n<|assistant|>\n{cleaned_assistant_response}"

print("--- Cleaned Full Model Output (Input + Response) ---")
print(final_full_output)

print("\n--- Extracted Assistant's Response ---")
print(cleaned_assistant_response)

# **Step 8: Fine-Tuning**

In [ ]:
# Load Dataset
medical_dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en", split = "train[:500]")


In [ ]:
medical_dataset[1]

In [ ]:
EOS_TOKEN = tokenizer.eos_token
EOS_TOKEN

In [ ]:
#Update the prompt style for training
train_prompt_style = """
Below is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response that effectively addresses the request.

Before crafting your answer, take a moment to carefully analyze the question. Develop a clear, step-by-step thought process to ensure your response is both logical and accurate.

### Instructions:
You are a medical expert specializing in clinical reasoning, diagnostics, and treatment planning. Answer the medical question below using your advanced knowledge.

### Question:
{}

### Response:
<think>
{}
</think>
{}
"""

In [ ]:
train_prompt_style = """<|user|>
{}

<|assistant|>
<think>
{}
</think>

{}
"""

In [ ]:
def preprocess_input_data(examples):
    inputs = examples["Question"]
    cots = examples["Complex_CoT"]
    outputs = examples["Response"]

    texts = []

    for inp, cot, out in zip(inputs, cots, outputs):
        text = train_prompt_style.format(inp, cot, out) + EOS_TOKEN
        texts.append(text)

    return {
        "text": texts
    }

In [ ]:
finetune_dataset = medical_dataset.map(
    preprocess_input_data,
    batched=True
)

In [ ]:
print(finetune_dataset.column_names)

In [ ]:
finetune_dataset["text"][0]

# **Step 9: Applying Lora Fine-Tuning to the Model**

In [ ]:
model_lora = FastLanguageModel.get_peft_model(
    model = model,
    r = 8,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_alpha = 16, #lora_alpha indicated how much the adapters influence the model output.
    lora_dropout = 0, #means no dropout will be applied to model layers.
    bias = "none",
    use_gradient_checkpointing= "unsloth",
    random_state = 3047,
    use_rslora= False,
    loftq_config=None
)

In [ ]:
trainer = SFTTrainer(
    model = model_lora,
    tokenizer = tokenizer,
    train_dataset = finetune_dataset,
    dataset_text_field = "text",
    max_seq_length = 1024,
    dataset_num_proc = 1,

    # Define Training Arguments
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        num_train_epochs = 1,
        warmup_steps = 5,
        max_steps = 200,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        report_to="wandb",
        logging_steps = 5,
        logging_strategy = "steps",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir = "outputs",
    ),

)

**Clearing Memory**

In [ ]:
import torch
torch.cuda.empty_cache()

**Setting WanDB**

In [ ]:
!pip install weave

In [ ]:
# Setup WANDB
# !pip install weave
import weave
from google.colab import userdata
wnb_token = userdata.get("WANDB_API_TOKEN")
# Login to WnB
wandb.login(key=wnb_token) # import wandb
run = wandb.init(
    project='Fine-tune-DeepSeek-R1-on-Medical-CoT-Dataset',
    job_type="training",
    anonymous="allow"
)

In [ ]:
torch.compile = False

In [ ]:
trainer_stats = trainer.train()

# **Step 10: Testing after Fine-Tuning**

In [ ]:
def format_prompt(question):
    return f"""<|user|>
{question}

<|assistant|>
"""

In [ ]:
def generate_response(question):
    prompt = format_prompt(question)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=300,
            do_sample=True,
            temperature=0.6,     # 🔥 lower = more accurate
            top_p=0.9,
            repetition_penalty=1.1,  # 🔥 reduces nonsense
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False
        )

    response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
    )

    # Extra cleanup
    response = response.replace("Ġ", " ")

    return response

In [ ]:
question = """A 61-year-old woman with urine leakage during coughing and sneezing but not at night. What would cystometry show?"""

response = generate_response(question)

print(response)